In [1]:
from gevent import monkey
monkey.patch_all(thread=False)
from gevent.queue import Queue
import gevent

import sys
sys.path.append('C:/Users/YangTeng/Desktop/venv/00.module')
import UA
import Proxy

# = = = = = = = = = = = = = = =

import pandas as pd

df = pd.read_excel('C:/Users/YangTeng/Desktop/文档/爬虫/blower motor resistor-detail-RockAuto.xlsx',
                   header=0,
                   dtype=str).fillna('')

list_type_code = df['Type_Code'].tolist()
list_part_key = df['Part_Key'].tolist()

print('总数量：' + str(len(list_part_key)))
print()

# = = = = = = = = = = = = = = =

df = pd.DataFrame()

# = = = = = = = = = = = = = = =

from urllib.parse import quote
import requests
import json

from bs4 import BeautifulSoup

from lxml import etree

# = = = = = = = = = = = = = = =

import time
from datetime import datetime

# = = = = = = = = = = = = = = =

work = Queue()
for type_code, part_key in zip(list_type_code, list_part_key):
    work.put_nowait((type_code, part_key))

def crawler():
    global df
    
    while not work.empty():
        type_code, part_key = work.get_nowait()
        
        # = = = = = = = = = = = = = = =
        
        status = 'error'
        for _ in range(30):
            try:
                if part_key == '':
                        df_temp = pd.DataFrame([{'status': 'ok',
                                                 'Part_Key': part_key,
                                                 'Vehicle': ''}])

                        status = 'ok'

                        break
                
                data = 'func=getbuyersguide&payload=' + quote('{"partData":{"listing_data_essential":{"parttype":"') + type_code + quote('","partkey":"') + part_key + quote('"}}}') + '&api_json_request=1&sctchecked=1&scbeenloaded=true&curCartGroupID='

                ua = UA.get_User_Agent_Requests()
                ua['Content-Type'] = 'application/x-www-form-urlencoded; charset=UTF-8'
                resp = requests.post('https://www.rockauto.com/catalog/catalogapi.php',
                                     data=data,
                                     headers=ua,
                                     proxies=Proxy.get_Proxy_Requests()).text

                resp = json.loads(resp)
                
                # = = = = = = = = = = = = = = =
                
                html = resp['buyersguidepieces']['body']

                soup = BeautifulSoup(html, 'lxml')
                
                # = = = = = = = = = = = = = = =
                
                html = etree.HTML(str(soup))

                list_vehicle = html.xpath('//div[contains(@class, "buyersguide-nested")]/descendant::table[contains(@class, "nobmp")]/tr')

                vehicle = '\n'.join([' '.join(_.xpath('./td/text()')).strip() for _ in list_vehicle])

                # = = = = = = = = = = = = = = =
           
                df_temp = pd.DataFrame([{'status': 'ok',
                                         'Part_Key': part_key,
                                         'Vehicle': vehicle}])
                
                status = 'ok'
                
                break
                
            except:
                time.sleep(0.5)
                continue

        if status == 'error':
            df_temp = pd.DataFrame([{'status': 'error',
                                     'Part_Key': part_key}])

        df = pd.concat([df, df_temp], ignore_index=True).fillna('')

        print(part_key +  '  <->  [' + status + '] - 剩余数量：' + str(work.qsize()) + ' - 当前时间：' + datetime.now().strftime('%H:%M:%S'))

# = = = = = = = = = = = = = = =

list_task = []
for _ in range(15):
    task = gevent.spawn(crawler)
    list_task.append(task)
gevent.joinall(list_task)

df = df.drop_duplicates(ignore_index=True)

print()
print('爬虫输出中')
df.to_excel('C:/Users/YangTeng/Desktop/文档/爬虫/blower motor resistor-detail-RockAuto-车型.xlsx_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '.xlsx', index=False)
print()
print('爬虫结束')

总数量：671

663452  <->  [ok] - 剩余数量：656 - 当前时间：09:58:28
445372  <->  [ok] - 剩余数量：655 - 当前时间：09:58:28
443206  <->  [ok] - 剩余数量：654 - 当前时间：09:58:28
13957617  <->  [ok] - 剩余数量：653 - 当前时间：09:58:28
13957629  <->  [ok] - 剩余数量：652 - 当前时间：09:58:28
13957621  <->  [ok] - 剩余数量：651 - 当前时间：09:58:28
14872581  <->  [ok] - 剩余数量：650 - 当前时间：09:58:28
14872589  <->  [ok] - 剩余数量：649 - 当前时间：09:58:28
14872577  <->  [ok] - 剩余数量：648 - 当前时间：09:58:28
663451  <->  [ok] - 剩余数量：647 - 当前时间：09:58:29
14640409  <->  [ok] - 剩余数量：646 - 当前时间：09:58:29
13957625  <->  [ok] - 剩余数量：645 - 当前时间：09:58:29
14872593  <->  [ok] - 剩余数量：644 - 当前时间：09:58:29
15197201  <->  [ok] - 剩余数量：643 - 当前时间：09:58:29
2415956  <->  [ok] - 剩余数量：642 - 当前时间：09:58:30
445461  <->  [ok] - 剩余数量：641 - 当前时间：09:58:30
445893  <->  [ok] - 剩余数量：640 - 当前时间：09:58:30
15197205  <->  [ok] - 剩余数量：639 - 当前时间：09:58:30
2416460  <->  [ok] - 剩余数量：638 - 当前时间：09:58:30
443393  <->  [ok] - 剩余数量：637 - 当前时间：09:58:30
3777747  <->  [ok] - 剩余数量：636 - 当前时间：09:58:30
14052225  <->  [ok] -

1319628  <->  [ok] - 剩余数量：476 - 当前时间：09:59:07
1320747  <->  [ok] - 剩余数量：475 - 当前时间：09:59:08
12392989  <->  [ok] - 剩余数量：474 - 当前时间：09:59:08
1320093  <->  [ok] - 剩余数量：473 - 当前时间：09:59:08
1320499  <->  [ok] - 剩余数量：472 - 当前时间：09:59:08
12392993  <->  [ok] - 剩余数量：471 - 当前时间：09:59:08
445737  <->  [ok] - 剩余数量：470 - 当前时间：09:59:08
1319540  <->  [ok] - 剩余数量：469 - 当前时间：09:59:09
1319580  <->  [ok] - 剩余数量：468 - 当前时间：09:59:09
10533988  <->  [ok] - 剩余数量：467 - 当前时间：09:59:09
13081333  <->  [ok] - 剩余数量：466 - 当前时间：09:59:09
1320174  <->  [ok] - 剩余数量：465 - 当前时间：09:59:09
1355013  <->  [ok] - 剩余数量：464 - 当前时间：09:59:10
1320645  <->  [ok] - 剩余数量：463 - 当前时间：09:59:10
1321168  <->  [ok] - 剩余数量：462 - 当前时间：09:59:11
1514010  <->  [ok] - 剩余数量：461 - 当前时间：09:59:11
1321022  <->  [ok] - 剩余数量：460 - 当前时间：09:59:11
10533992  <->  [ok] - 剩余数量：459 - 当前时间：09:59:11
3778470  <->  [ok] - 剩余数量：458 - 当前时间：09:59:12
1354537  <->  [ok] - 剩余数量：457 - 当前时间：09:59:13
1320105  <->  [ok] - 剩余数量：456 - 当前时间：09:59:13
1320845  <->  [ok] - 剩余数量：455 

10461892  <->  [ok] - 剩余数量：297 - 当前时间：10:00:03
1513684  <->  [ok] - 剩余数量：296 - 当前时间：10:00:04
4693161  <->  [ok] - 剩余数量：295 - 当前时间：10:00:04
3787047  <->  [ok] - 剩余数量：294 - 当前时间：10:00:05
3770166  <->  [ok] - 剩余数量：293 - 当前时间：10:00:05
2413244  <->  [ok] - 剩余数量：292 - 当前时间：10:00:05
3781896  <->  [ok] - 剩余数量：291 - 当前时间：10:00:05
3776007  <->  [ok] - 剩余数量：290 - 当前时间：10:00:05
3769155  <->  [ok] - 剩余数量：289 - 当前时间：10:00:05
320091  <->  [ok] - 剩余数量：288 - 当前时间：10:00:06
3788949  <->  [ok] - 剩余数量：287 - 当前时间：10:00:06
3774093  <->  [ok] - 剩余数量：286 - 当前时间：10:00:06
4689453  <->  [ok] - 剩余数量：285 - 当前时间：10:00:06
10461896  <->  [ok] - 剩余数量：284 - 当前时间：10:00:06
3769428  <->  [ok] - 剩余数量：283 - 当前时间：10:00:07
320092  <->  [ok] - 剩余数量：282 - 当前时间：10:00:07
3777957  <->  [ok] - 剩余数量：281 - 当前时间：10:00:08
3784248  <->  [ok] - 剩余数量：280 - 当前时间：10:00:08
6439653  <->  [ok] - 剩余数量：279 - 当前时间：10:00:08
3784272  <->  [ok] - 剩余数量：278 - 当前时间：10:00:09
10461904  <->  [ok] - 剩余数量：277 - 当前时间：10:00:10
13081357  <->  [ok] - 剩余数量：276 - 

3786987  <->  [ok] - 剩余数量：118 - 当前时间：10:00:55
446058  <->  [ok] - 剩余数量：117 - 当前时间：10:00:56
6439695  <->  [ok] - 剩余数量：116 - 当前时间：10:00:56
6439701  <->  [ok] - 剩余数量：115 - 当前时间：10:00:56
443778  <->  [ok] - 剩余数量：114 - 当前时间：10:00:56
6439716  <->  [ok] - 剩余数量：113 - 当前时间：10:00:57
6588280  <->  [ok] - 剩余数量：112 - 当前时间：10:00:58
6439704  <->  [ok] - 剩余数量：111 - 当前时间：10:00:58
6086706  <->  [ok] - 剩余数量：110 - 当前时间：10:00:59
6439692  <->  [ok] - 剩余数量：109 - 当前时间：10:00:59
6505512  <->  [ok] - 剩余数量：108 - 当前时间：10:00:59
6505515  <->  [ok] - 剩余数量：107 - 当前时间：10:00:59
445735  <->  [ok] - 剩余数量：106 - 当前时间：10:01:00
6439707  <->  [ok] - 剩余数量：105 - 当前时间：10:01:00
6588308  <->  [ok] - 剩余数量：104 - 当前时间：10:01:00
6588284  <->  [ok] - 剩余数量：103 - 当前时间：10:01:00
6994972  <->  [ok] - 剩余数量：102 - 当前时间：10:01:01
6588296  <->  [ok] - 剩余数量：101 - 当前时间：10:01:02
6994980  <->  [ok] - 剩余数量：100 - 当前时间：10:01:02
6994976  <->  [ok] - 剩余数量：99 - 当前时间：10:01:02
6086685  <->  [ok] - 剩余数量：98 - 当前时间：10:01:02
6439719  <->  [ok] - 剩余数量：97 - 当前时间：10:

C:\Users\YangTeng\Desktop\venv\lib\site-packages\gevent\hub.py:161: UserWarning: libuv only supports millisecond timer resolution; all times less will be set to 1 ms
  with loop.timer(seconds, ref=ref) as t:
